# HLO ontology training
Extract the project ZIP under `/content` in Colab and set `PROJECT` to the extracted project root. Enable a GPU and set `TRAIN_MODEL = True` for full training. The default runs data checks, unit tests and the SPARQL baseline.
Set WANDB_MODE to 'offline' for local tracking or 'online' to publish metrics. For online tracking, add WANDB_API_KEY to Colab Secrets and grant access to this notebook.


In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path('/content/HLO-recommendation-system-main')
OUTPUT = Path('/content/hlo-runs')
TRAIN_MODEL = False

WANDB_MODE = 'offline'
WANDB_PROJECT = 'hlo-recommendation'
WANDB_ENTITY = None


In [ ]:
runner = PROJECT / 'tools/train_hlo.py'
if not runner.is_file():
    raise FileNotFoundError(f'Extract the project ZIP and set PROJECT: {runner}')
command = [sys.executable, '-u', str(runner), '--project', str(PROJECT), '--output', str(OUTPUT)]
if not TRAIN_MODEL:
    command.append('--check-only')
command.extend(['--wandb-mode', WANDB_MODE, '--wandb-project', WANDB_PROJECT])
if WANDB_ENTITY:
    command.extend(['--wandb-entity', WANDB_ENTITY])
if WANDB_MODE != 'disabled':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[tracking]'], cwd=PROJECT, check=True)
if WANDB_MODE == 'online':
    import os
    from google.colab import userdata
    if not os.environ.get('WANDB_API_KEY'):
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
subprocess.run(command, cwd=PROJECT, check=True)
